[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/03_cross_entropy_and_loss_functions/first_principles.ipynb)

# Topic 03: Cross-Entropy and Loss Functions

## 1. First-Principles Intuition & Motivation

Entropy (Topic 01) measured the cost of describing reality when we *know* its distribution.
But models never know the true distribution — they carry an approximation $q$ of an unknown $p$.
Cross-entropy asks the practical question: **what does it cost to act on $q$ when reality follows $p$?**

The coding story makes this concrete.
An optimal code for $q$ assigns the symbol $x$ a codeword of length $-\log q(x)$ bits.
If symbols actually arrive with frequencies $p(x)$, the *realized* average code length is

$$
H(p, q) = -\sum_x p(x) \log q(x)
$$

— the expectation under reality of the cost schedule designed for the model.

### The Fundamental Decomposition

Adding and subtracting the entropy of $p$ splits the bill into two invoices:

$$
H(p, q) = \underbrace{H(p)}_{\text{price of reality}} + \underbrace{D_{\mathrm{KL}}(p \parallel q)}_{\text{price of being wrong}}
$$

The first term is beyond anyone's control: it is the intrinsic unpredictability of the source.
The second term is entirely the modeler's fault, and Gibbs' inequality (Proof 3.1) shows it is always nonnegative and vanishes exactly when $q = p$.

This is why cross-entropy is a legitimate training objective: *pushing $H(p, q)$ down can only be done by moving $q$ toward $p$*, because the floor $H(p)$ is fixed.

### Three Faces of the Same Quantity

| Community | Name | Expression |
|---|---|---|
| Information theory | expected code length under the wrong code | $-\sum_x p(x)\log_2 q(x)$ bits |
| Statistics | negative expected log-likelihood | $-\mathbb{E}_{p}\left[\ln q(X)\right]$ nats |
| Machine learning | cross-entropy loss | $-\frac{1}{N}\sum_i \log q(y_i \mid x_i)$ |

The third row is the empirical version of the second: replacing $p$ by the empirical distribution of the data turns expected cross-entropy into the training loss, and maximum likelihood estimation into cross-entropy minimization.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Cross-Entropy)

For distributions $p, q$ on a common alphabet $\mathcal{X}$ with $q(x) \gt 0$ wherever $p(x) \gt 0$ (absolute continuity),

$$
H(p, q) = -\sum_{x \in \mathcal{X}} p(x) \log q(x) = \mathbb{E}_{X \sim p}\left[-\log q(X)\right]
$$

If some $x$ has $p(x) \gt 0$ but $q(x) = 0$, then $H(p, q) = +\infty$: a model that declares a possible event impossible pays an infinite bill the first time the event occurs.

### Definition 2.2 (Binary Cross-Entropy / Log Loss)

For a label $y \in \{0, 1\}$ and predicted probability $\hat{p} \in (0, 1)$,

$$
\mathrm{BCE}(y, \hat{p}) = -y \log \hat{p} - (1 - y)\log(1 - \hat{p})
$$

Averaged over data, this is the log loss of Kaggle leaderboards and the negative Bernoulli log-likelihood of logistic regression.

### Definition 2.3 (Softmax and the Softmax Cross-Entropy Loss)

Given logits $z \in \mathbb{R}^K$, the softmax distribution is

$$
q_k = \mathrm{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
$$

and the loss against a target distribution $y$ (one-hot or soft) is $\mathcal{L}(z) = -\sum_k y_k \log q_k$.

### Definition 2.4 (Proper Scoring Rule)

A scoring rule $S(q, x)$ assigns a loss when $x$ occurs and $q$ was reported.
It is **proper** if honest reporting minimizes expected loss: for all $p, q$,

$$
\mathbb{E}_{x \sim p}\left[S(p, x)\right] \le \mathbb{E}_{x \sim p}\left[S(q, x)\right]
$$

and **strictly proper** if equality forces $q = p$.
It is **local** if $S(q, x)$ depends on $q$ only through the value $q(x)$.

### Theorem Statements

- **Theorem A (Gibbs' inequality)**: $H(p, q) \ge H(p)$, equivalently $D_{\mathrm{KL}}(p \parallel q) \ge 0$, with equality iff $p = q$.
- **Theorem B (MLE–CE equivalence)**: minimizing empirical cross-entropy over a model family equals maximizing the likelihood of the data over that family.
- **Theorem C (Softmax gradient)**: for $\mathcal{L}(z) = -\sum_k y_k \log \mathrm{softmax}(z)_k$ with $\sum_k y_k = 1$: $\frac{\partial \mathcal{L}}{\partial z_k} = q_k - y_k$.
- **Theorem D (Log score propriety)**: $S(q, x) = -\log q(x)$ is strictly proper; moreover, up to affine transformation it is the *only* smooth, local, strictly proper scoring rule on alphabets with 3 or more outcomes (Bernardo, 1979).
- **Theorem E (Label smoothing optimum)**: against smoothed targets $\tilde{y} = (1 - \epsilon)y + \epsilon u$ (uniform $u$), the loss-minimizing prediction is $q^* = \tilde{y}$, keeping optimal confidence at $1 - \epsilon + \epsilon/K$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1 (Gibbs' inequality: $H(p, q) \ge H(p)$)

**Step 1.** Compute the gap:

$$
H(p, q) - H(p) = \sum_x p(x)\log\frac{p(x)}{q(x)} = D_{\mathrm{KL}}(p \parallel q)
$$

**Step 2.** Use the tangent-line bound $\ln t \le t - 1$ for all $t \gt 0$, with equality iff $t = 1$.
Apply it with $t = q(x)/p(x)$ on the support of $p$:

$$
-D_{\mathrm{KL}}(p \parallel q) = \sum_x p(x)\ln\frac{q(x)}{p(x)} \le \sum_x p(x)\left(\frac{q(x)}{p(x)} - 1\right) = \sum_x q(x) - \sum_x p(x) \le 1 - 1 = 0
$$

**Step 3.** Hence $D_{\mathrm{KL}}(p \parallel q) \ge 0$, i.e., $H(p, q) \ge H(p)$.
Equality requires $q(x)/p(x) = 1$ at every support point of $p$ and $\sum_x q(x) = 1$ concentrated there, i.e., $p = q$.

$$
\boxed{H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q) \ge H(p), \quad \text{equality} \iff p = q}
$$

$\blacksquare$

### Proof 3.2 (Minimizing cross-entropy = maximum likelihood)

**Step 1.** Let data $y_1, \dots, y_N$ be drawn from $p$, and let $\hat{p}_N$ denote their empirical distribution ($\hat{p}_N(x)$ = fraction of samples equal to $x$).
The log-likelihood of a model $q_\theta$ is

$$
\ell(\theta) = \sum_{i=1}^{N} \log q_\theta(y_i) = N \sum_{x} \hat{p}_N(x) \log q_\theta(x) = -N \, H(\hat{p}_N, q_\theta)
$$

**Step 2.** Therefore $\arg\max_\theta \ell(\theta) = \arg\min_\theta H(\hat{p}_N, q_\theta)$: the MLE is exactly the cross-entropy minimizer against the empirical distribution.

**Step 3.** Since $H(\hat{p}_N, q) = H(\hat{p}_N) + D_{\mathrm{KL}}(\hat{p}_N \parallel q)$ and the first term is $\theta$-free, MLE equivalently minimizes $D_{\mathrm{KL}}(\hat{p}_N \parallel q_\theta)$ — maximum likelihood is *forward* KL projection of the data onto the model family.

$$
\boxed{\text{MLE} = \arg\min_\theta H(\hat{p}_N, q_\theta) = \arg\min_\theta D_{\mathrm{KL}}(\hat{p}_N \parallel q_\theta)}
$$

$\blacksquare$

### Proof 3.3 (The softmax cross-entropy gradient $q - y$)

**Step 1.** Write the loss with the log-sum-exp normalizer $\mathrm{LSE}(z) = \log\sum_j e^{z_j}$:

$$
\mathcal{L}(z) = -\sum_k y_k \log q_k = -\sum_k y_k \left(z_k - \mathrm{LSE}(z)\right) = -\sum_k y_k z_k + \mathrm{LSE}(z)
$$

using $\sum_k y_k = 1$ to pull the normalizer out of the sum.

**Step 2.** Differentiate the two pieces with respect to $z_m$.
The linear piece gives $-y_m$; the normalizer gives

$$
\frac{\partial}{\partial z_m}\mathrm{LSE}(z) = \frac{e^{z_m}}{\sum_j e^{z_j}} = q_m
$$

**Step 3.** Combine:

$$
\boxed{\frac{\partial \mathcal{L}}{\partial z_m} = q_m - y_m}
$$

**Interpretation.** The backpropagated error is literally *predicted minus target probability*: bounded in $[-1, 1]$, zero exactly at a perfect match, and free of any $\sigma'$-style attenuation factor.
This identity — the derivative of LSE is softmax — is why the pair is numerically and geometrically matched (LSE is the convex conjugate log-partition function of the categorical exponential family). $\blacksquare$

### Proof 3.4 (Strict propriety of the logarithmic score)

**Claim.** Reporting $q$ when the truth is $p$ has expected log loss $\mathbb{E}_p\left[-\log q(X)\right] = H(p, q)$, minimized uniquely at $q = p$.

**Proof.** By Gibbs' inequality (Proof 3.1),

$$
\mathbb{E}_p\left[-\log q(X)\right] = H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q) \ge H(p) = \mathbb{E}_p\left[-\log p(X)\right]
$$

with equality iff $q = p$. Hence a forecaster judged by log loss can do no better than announce exact beliefs — the score is strictly proper.

**Remark (locality).** The log score uses only $q(x)$ at the realized outcome, and Bernardo (1979) showed that on alphabets of size $\ge 3$ every smooth, local, strictly proper rule is $a - b\log q(x)$ with $b \gt 0$: locality essentially *forces* the logarithm, and with it the whole apparatus of entropy.

$$
\boxed{\text{log loss is strictly proper; honesty is the unique optimum}}
$$

$\blacksquare$

### Proof 3.5 (Label smoothing: decomposition and optimal prediction)

**Setup.** Smooth a one-hot target $y$ toward uniform $u_k = 1/K$: $\tilde{y} = (1 - \epsilon)y + \epsilon u$. The loss is $\mathcal{L} = H(\tilde{y}, q)$.

**Step 1 (decomposition).** By linearity of cross-entropy in its first argument:

$$
H(\tilde{y}, q) = (1 - \epsilon) H(y, q) + \epsilon H(u, q)
$$

so smoothing adds a fraction of the "code the uniform distribution with $q$" cost — a penalty on distributions far from uniform, since $H(u, q) = \log K + D_{\mathrm{KL}}(u \parallel q)$.

**Step 2 (optimal $q$).** Minimize $H(\tilde{y}, q) = H(\tilde{y}) + D_{\mathrm{KL}}(\tilde{y} \parallel q)$ over $q$: Gibbs' inequality gives the unique minimizer $q^* = \tilde{y}$.

**Step 3 (optimal confidence and logit gap).** The optimal predicted probability of the true class is

$$
q^*_{\text{true}} = 1 - \epsilon + \frac{\epsilon}{K}
$$

and the optimal logit gap between the true class and any other is finite: $z_{\text{true}} - z_{\text{other}} = \log\frac{(1-\epsilon)K + \epsilon}{\epsilon}$, instead of diverging to infinity as with hard targets.

$$
\boxed{q^* = (1-\epsilon)y + \epsilon u; \quad \text{confidence capped at } 1 - \epsilon + \epsilon/K}
$$

**Interpretation.** Label smoothing converts an unattainable optimum (infinite logits) into an interior one, improving calibration and gradient conditioning. $\blacksquare$

### Proof 3.6 (Why MSE fails and CE succeeds with sigmoid outputs)

**Setup.** A single sigmoid unit $\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$ predicts label $y \in \{0, 1\}$.

**Step 1 (MSE gradient).** For $\mathcal{L}_{\mathrm{MSE}} = \tfrac{1}{2}(\hat{p} - y)^2$, the chain rule gives

$$
\frac{\partial \mathcal{L}_{\mathrm{MSE}}}{\partial z} = (\hat{p} - y)\, \sigma'(z) = (\hat{p} - y)\, \hat{p}(1 - \hat{p})
$$

At a *confidently wrong* prediction ($y = 1$, $\hat{p} \to 0$), the factor $\hat{p}(1 - \hat{p}) \to 0$ crushes the gradient: the worse the mistake, the weaker the learning signal.

**Step 2 (CE gradient).** For $\mathcal{L}_{\mathrm{BCE}} = -y\log\hat{p} - (1-y)\log(1-\hat{p})$:

$$
\frac{\partial \mathcal{L}_{\mathrm{BCE}}}{\partial z} = \frac{\hat{p} - y}{\hat{p}(1 - \hat{p})} \cdot \hat{p}(1 - \hat{p}) = \hat{p} - y
$$

The sigmoid's vanishing derivative cancels exactly against the loss's growing derivative, leaving a gradient proportional to the raw error.

**Step 3 (convexity).** $\mathcal{L}_{\mathrm{BCE}}(z)$ is convex in $z$ (its second derivative is $\hat{p}(1-\hat{p}) \gt 0$), whereas $\mathcal{L}_{\mathrm{MSE}}(z)$ is non-convex in $z$ with flat plateaus at both tails.

$$
\boxed{\text{CE: } \partial_z \mathcal{L} = \hat{p} - y \text{ (convex, no plateaus)}; \quad \text{MSE: gradient vanishes when confidently wrong}}
$$

$\blacksquare$

## 4. Computational & Algorithmic Insights

### Never Exponentiate Then Take Logs

The naive pipeline logits $\to$ softmax $\to$ log invites overflow ($e^{800} = \infty$) and catastrophic cancellation ($\log$ of a probability rounded to 0 gives $-\infty$).
Frameworks therefore fuse the operations:

$$
\log q_k = z_k - \mathrm{LSE}(z), \qquad \mathrm{LSE}(z) = m + \log\sum_j e^{z_j - m}, \quad m = \max_j z_j
$$

- PyTorch: `nn.CrossEntropyLoss` consumes raw logits (`log_softmax` + `nll_loss` fused); `BCEWithLogitsLoss` fuses sigmoid + BCE via the stable form $\max(z, 0) - zy + \log(1 + e^{-\vert z \vert})$.
- Passing already-softmaxed probabilities into a loss that re-applies `log_softmax` is a classic silent bug: the model still trains, but toward the wrong optimum.

### Loss Bookkeeping: Units, Floors, Reductions

- **Units**: framework losses are in nats; divide by $\ln 2$ for bits. A 10-class uniform guesser scores $\ln 10 \approx 2.303$ nats — always compare initial loss to $\log K$ as a sanity check.
- **Floor**: the expected loss of the *best possible* model is $H(Y \mid X)$ (Topic 02); persistent loss above it measures the model's KL gap, and training loss below it on finite data signals memorization.
- **Reduction**: token-averaged vs sequence-summed losses differ by length weighting; perplexity must be computed as $\exp$ of the *token-level mean* NLL.
- **Class imbalance**: weighted CE $-w_{y}\log q_{y}$ rescales gradients per class; focal loss $-(1 - q_y)^{\gamma}\log q_y$ instead down-weights *easy* examples adaptively.

## 5. Real-World Physics & AI/ML Applications

### Language Models: the Loss You Read About Is Cross-Entropy

Pretraining loss curves of GPT-class models plot exactly $H(\hat{p}_{\text{data}}, q_\theta)$ per token.
Scaling laws (Kaplan et al., 2020; Hoffmann et al., 2022) fit this cross-entropy as a power law in parameters and data plus an irreducible constant — an empirical estimate of the entropy rate of text.
Evaluation reports the same quantity as perplexity ($e^{\mathrm{CE}}$) or bits-per-byte ($\mathrm{CE}/\ln 2$ per byte).

### Classification, Calibration, and Distillation

- **Logistic regression** is BCE minimization; its gradient $\left(\sigma(z) - y\right)x$ is Proof 3.6 with features attached.
- **Calibration**: because log loss is strictly proper, a fully minimized CE would yield calibrated probabilities; overfit networks become miscalibrated, and temperature scaling re-minimizes CE on held-out data over a single scalar.
- **Knowledge distillation** replaces one-hot targets with a teacher's tempered softmax: the student minimizes $H(p_{\text{teacher}}^{(T)}, q_{\text{student}}^{(T)})$, transferring "dark knowledge" carried in the teacher's non-argmax probabilities.
- **Label smoothing** (Proof 3.5) is distillation from a maximally ignorant teacher blended with the ground truth.

### Physics and Gambling: Codes, Bets, and Free Energy

Kelly gambling makes cross-entropy tangible: betting fractions $q$ on outcomes distributed as $p$ grows wealth at rate $\log$-capital $= -H(p, q)$ per round (up to the odds), so the KL gap is literally lost money per bet.
In statistical mechanics, minimizing variational free energy $F(q) = \mathbb{E}_q[E] - T\,H(q)$ balances an energy (cross-entropy-like) term against entropy — the same trade-off that reappears as the ELBO in Topic 06.

$$
\text{growth-rate loss} = D_{\mathrm{KL}}(p \parallel q) \quad \text{— the gambler's KL tax}
$$

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Wrong-code penalty, source coding | Shannon (1948); Cover & Thomas (2006), Chapter 5 |
| Gibbs' inequality, KL nonnegativity | Cover & Thomas (2006), Section 2.6 |
| Log score and rational decisions | Good (1952); Bernardo (1979), *Expected Information as Expected Utility* |
| Proper scoring rules survey | Gneiting & Raftery (2007), JASA |
| MLE = CE minimization, softmax | Goodfellow, Bengio & Courville (2016), Chapters 5–6 |
| Label smoothing | Szegedy et al. (2016); Müller, Kornblith & Hinton (2019), *When Does Label Smoothing Help?* |
| Focal loss | Lin et al. (2017), ICCV |
| Distillation | Hinton, Vinyals & Dean (2015) |
| Kelly gambling and log-optimal growth | Kelly (1956); Cover & Thomas (2006), Chapter 6 |
| LM scaling laws (CE as the scaled quantity) | Kaplan et al. (2020); Hoffmann et al. (2022) |